# ICT — Annexe : la contextualité du zoo de proxys est un CSP

**Navigation** : [Index](README.md) | [Version Python](ICT-Annexe-ProxyContextuality-Python.ipynb)

> **Annexe spéculative, non numérotée.** Registre : issue #7290, dans l'épic ICT #4588.
> Cette annexe *instruit* un transfert conceptuel — elle ne le proclame pas. Le résultat
> qu'elle établit est **négatif**, et c'est précisément son intérêt.

## La question

Le « zoo de proxys » de la série ICT mesure une même trajectoire par plusieurs
instruments (écart spectral, sensibilité moyenne, sensibilité maximale), sous
plusieurs **grains de description** (*coarse-grainings*). Deux instruments peuvent
être en désaccord. La question de cette annexe :

> Ce désaccord est-il un simple **désaccord de valeurs** — chaque instrument mesure
> quelque chose de légèrement différent — ou une **obstruction structurelle**, au sens
> où *aucune* attribution de valeurs cohérente n'existe globalement, bien que chaque
> instrument soit localement parfaitement cohérent ?

La seconde forme porte un nom en mécanique quantique : c'est la **contextualité** au sens
de Kochen–Specker, formalisée par Abramsky et Brandenburger dans un cadre de faisceaux.
Elle a une propriété rare pour un concept importé : **elle se calcule**. Nous allons voir
qu'elle se ramène littéralement à un problème de satisfaction de contraintes (CSP).

## Le plan

1. Traduire le vocabulaire terme à terme, et **valider l'instrument sur un cas dont la
   réponse est prouvée à la main** avant de le croire sur des données.
2. Voir qu'il y a **trois crans**, pas deux — un `SAT` nu ne dit pas « non contextuel ».
3. Installer le **garde-fou** : trois manières pour un verdict d'être vide.
4. Mesurer sur le substrat S1, et rendre un verdict **honnête**.

> **Statut épistémique** — **Sans verdict à ce jour** : aucune ligne de la [matrice de dissociations](../../../docs/ict/dissociations-matrix.md) ne concerne ce notebook ; son statut épistémique sera porté par la matrice le cas échéant.

## 1. Le dictionnaire

Le cadre d'Abramsky–Brandenburger se transpose terme à terme. C'est ce tableau qui rend
le transfert *instruit* plutôt que décoratif — chaque ligne est une correspondance qu'on
peut contester nommément.

| Cadre quantique (Kochen–Specker) | Zoo de proxys ICT |
|---|---|
| Observable | un proxy (`spectral_gap`, `sensitivity_mean`, …) |
| Contexte de mesure (observables co-mesurables) | un protocole : quels proxys sont calculés ensemble |
| Support de la distribution du contexte | les tuples d'issues **effectivement observés** |
| Section locale | une attribution de valeurs cohérente dans **un** contexte |
| Section globale | une attribution cohérente **simultanément dans tous** les contextes |
| Fortement contextuel | aucune section globale n'existe |

La dernière ligne est le cœur calculable : « une section globale existe-t-elle ? » est un
CSP à **contraintes extensionnelles** — une table de tuples autorisés par contexte. Dans
un solveur CSP cela s'écrit directement comme une table de tuples autorisés par contexte
(`AddAllowedAssignments` en CP-SAT).

### Valider le détecteur avant de le croire

On ne braque pas un instrument neuf sur des données inconnues. On le braque d'abord sur
un cas dont la réponse est **prouvée à la main** : la **boîte de Popescu–Rohrlich**, objet
canonique fortement contextuel. Son insatisfiabilité se démontre par parité — les quatre
contraintes XOR somment à

$$2\,(a_1 + a_2 + b_1 + b_2) \equiv 1 \pmod 2$$

dont le membre de gauche est pair et celui de droite impair. Aucune attribution ne peut
donc satisfaire les quatre contextes ensemble.

In [1]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;
using System.Numerics;

// Illustration 1 — la boite de PR et l'obstruction minimale.
//
// Le module Python `ict/proxy_contextuality.py` est rejoue ici en BCL pur : un
// contexte est un support extensionnel, une section globale est une affectation
// dont la restriction a chaque contexte tombe dans le support de ce contexte.
// Deux voies independantes repondent a la meme question : l'enumeration
// exhaustive bornee (l'oracle deterministe, qui canonicalise les temoins) et
// une resolution par propagation (le solveur CSP, qui ne rend aucun temoin).

public sealed class Contexte
{
    public string[] Proxies { get; }
    public int[][] Support { get; }

    public Contexte(string[] proxies, int[][] support)
    {
        if (proxies == null || proxies.Length == 0)
            throw new ArgumentException("un contexte doit porter au moins un proxy");
        if (proxies.Distinct(StringComparer.Ordinal).Count() != proxies.Length)
            throw new ArgumentException("proxys dupliques dans un contexte");
        if (support == null || support.Length == 0)
            throw new ArgumentException("support vide : un contexte non mesure n'est pas un contexte sans contrainte");
        foreach (var t in support)
            if (t.Length != proxies.Length)
                throw new ArgumentException("tuple d'arite incorrecte dans un contexte");
        Proxies = proxies;
        Support = support;
    }

    public int[] Restreint(int[] affectation, Dictionary<string, int> position)
        => Proxies.Select(p => affectation[position[p]]).ToArray();

    public bool Admet(int[] affectation, Dictionary<string, int> position)
    {
        var tup = Restreint(affectation, position);
        foreach (var s in Support)
        {
            bool egal = true;
            for (int j = 0; j < s.Length; j++)
                if (s[j] != tup[j]) { egal = false; break; }
            if (egal) return true;
        }
        return false;
    }
}

public sealed class ModeleEmpirique
{
    public List<Contexte> Contextes { get; }
    public int[] Issues { get; }
    public string[] Mesures { get; }
    public Dictionary<string, int> Position { get; }
    public BigInteger EspaceRecherche => BigInteger.Pow(Issues.Length, Mesures.Length);

    public ModeleEmpirique(List<Contexte> contextes, int[] issues)
    {
        Contextes = contextes;
        Issues = issues;
        var vus = new List<string>();
        foreach (var c in contextes)
            foreach (var p in c.Proxies)
                if (!vus.Contains(p)) vus.Add(p);
        Mesures = vus.OrderBy(p => p, StringComparer.Ordinal).ToArray();
        Position = new Dictionary<string, int>(StringComparer.Ordinal);
        for (int j = 0; j < Mesures.Length; j++) Position[Mesures[j]] = j;
    }

    public static ModeleEmpirique FromContexts(IEnumerable<Contexte> contextes, int[]? issues = null)
    {
        var liste = contextes.ToList();
        if (liste.Count == 0)
            throw new ArgumentException("un modele empirique doit contenir au moins un contexte");
        var observes = new SortedSet<int>();
        foreach (var c in liste)
            foreach (var t in c.Support)
                foreach (var v in t) observes.Add(v);
        int[] alphabet = (issues ?? observes.ToArray()).Distinct().OrderBy(o => o).ToArray();
        if (alphabet.Length < 2)
            throw new ArgumentException("alphabet d'issues trivial : toute assignation serait forcee");
        foreach (var v in observes)
            if (!alphabet.Contains(v))
                throw new ArgumentException("issues hors alphabet dans les supports");
        return new ModeleEmpirique(liste, alphabet);
    }
}

public sealed class RapportRecouvrement
{
    public Dictionary<string, int> Multiplicite { get; } = new Dictionary<string, int>(StringComparer.Ordinal);
    public List<string> ProxysPartages { get; } = new List<string>();
    public int NRecouvrementsDistincts { get; set; }
    public List<int> TaillesSupport { get; } = new List<int>();
    public int MinTailleSupport { get; set; }
    public int MaxTailleSupport { get; set; }
    public string? Degenerescence { get; set; }
    public string RaisonDegenerescence { get; set; } = "";
}

public sealed class ResultatVerdict
{
    public RapportRecouvrement Rapport { get; set; } = new RapportRecouvrement();
    public bool? Satisfiable { get; set; }
    public string Verdict { get; set; } = "";
    public string Rationale { get; set; } = "";
    public bool SolveurDisponible { get; set; }
    public bool? SolveurSatisfiable { get; set; }
    public string StatutSolveur { get; set; } = "";
    public int SectionsGlobales { get; set; }
    public int SectionsLocales { get; set; }
    public int SansExtension { get; set; }
    public int[]? ObstructionMinimale { get; set; }
}

public static class Csp
{
    public static RapportRecouvrement Rapport(ModeleEmpirique m)
    {
        var r = new RapportRecouvrement();
        foreach (var c in m.Contextes)
            foreach (var p in c.Proxies)
                r.Multiplicite[p] = r.Multiplicite.TryGetValue(p, out var n) ? n + 1 : 1;
        foreach (var kv in r.Multiplicite.OrderBy(kv => kv.Key, StringComparer.Ordinal))
            if (kv.Value >= 2) r.ProxysPartages.Add(kv.Key);
        foreach (var c in m.Contextes) r.TaillesSupport.Add(c.Support.Length);
        r.MinTailleSupport = r.TaillesSupport.Min();
        r.MaxTailleSupport = r.TaillesSupport.Max();
        var couvertures = new HashSet<string>(StringComparer.Ordinal);
        foreach (var c in m.Contextes)
            couvertures.Add(string.Join("|", c.Proxies.OrderBy(p => p, StringComparer.Ordinal)));
        r.NRecouvrementsDistincts = couvertures.Count;
        if (r.NRecouvrementsDistincts == 1)
        {
            r.Degenerescence = "degenerate_single_cover";
            r.RaisonDegenerescence = "tous les contextes co-mesurent le meme ensemble de proxys : le recouvrement est trivial et la question se reduit a l'intersection des supports (accord entre protocoles). La contextualite exige un recouvrement PARTIEL, donc des proxys non conjointement mesurables";
        }
        else if (r.ProxysPartages.Count == 0)
        {
            r.Degenerescence = "degenerate_no_overlap";
            r.RaisonDegenerescence = "aucun proxy n'est co-mesure dans deux contextes : les contextes sont independants, une section globale existe par construction";
        }
        else if (r.TaillesSupport.All(s => s == 1))
        {
            r.Degenerescence = "degenerate_rigid";
            r.RaisonDegenerescence = "tous les supports sont des singletons : un UNSAT signifierait seulement qu'un proxy change de valeur d'un contexte a l'autre (dissociation), pas une obstruction de structure";
        }
        return r;
    }

    public static List<int[]> SectionsGlobales(ModeleEmpirique m, int maxSearch = 1 << 22, int? limit = null)
    {
        if (m.EspaceRecherche > maxSearch)
            throw new OverflowException("espace de recherche " + m.EspaceRecherche + " > max_search " + maxSearch);
        var trouvees = new List<int[]>();
        int n = m.Mesures.Length;
        var combinaison = new int[n];
        void Explorer(int idx)
        {
            if (limit.HasValue && trouvees.Count >= limit.Value) return;
            if (idx == n)
            {
                if (Compatible(m, combinaison)) trouvees.Add((int[])combinaison.Clone());
                return;
            }
            foreach (var o in m.Issues)
            {
                combinaison[idx] = o;
                Explorer(idx + 1);
                if (limit.HasValue && trouvees.Count >= limit.Value) return;
            }
        }
        Explorer(0);
        return trouvees;
    }

    private static bool Compatible(ModeleEmpirique m, int[] affectation)
    {
        foreach (var ctx in m.Contextes)
            if (!ctx.Admet(affectation, m.Position)) return false;
        return true;
    }

    public static bool? ResoudreParPropagation(ModeleEmpirique m)
    {
        int n = m.Mesures.Length;
        var affectation = new int[n];
        var assignes = new bool[n];
        bool Resoudre(int idx)
        {
            if (idx == n) return true;
            foreach (var o in m.Issues)
            {
                affectation[idx] = o;
                assignes[idx] = true;
                if (PrefixeCompatible(m, affectation, assignes) && Resoudre(idx + 1)) return true;
                assignes[idx] = false;
            }
            return false;
        }
        return Resoudre(0);
    }

    private static bool PrefixeCompatible(ModeleEmpirique m, int[] affectation, bool[] assignes)
    {
        foreach (var ctx in m.Contextes)
        {
            bool auMoinsUn = false;
            foreach (var s in ctx.Support)
            {
                bool ok = true;
                for (int j = 0; j < ctx.Proxies.Length; j++)
                {
                    int idx = m.Position[ctx.Proxies[j]];
                    if (assignes[idx] && s[j] != affectation[idx]) { ok = false; break; }
                }
                if (ok) { auMoinsUn = true; break; }
            }
            if (!auMoinsUn) return false;
        }
        return true;
    }

    public static (int nLocal, int nSansExtension) SectionsLocales(ModeleEmpirique m, List<int[]> sections)
    {
        int nLocal = 0, nSans = 0;
        foreach (var ctx in m.Contextes)
            foreach (var tup in ctx.Support)
            {
                nLocal++;
                bool etendue = sections.Any(s => ctx.Restreint(s, m.Position).SequenceEqual(tup));
                if (!etendue) nSans++;
            }
        return (nLocal, nSans);
    }

    public static int[]? ObstructionMinimale(ModeleEmpirique m)
    {
        if (SectionsGlobales(m, limit: 1).Count > 0) return null;
        var keep = Enumerable.Range(0, m.Contextes.Count).ToList();
        bool changed = true;
        while (changed)
        {
            changed = false;
            foreach (int ci in keep.ToList())
            {
                if (keep.Count == 1) break;
                var trial = keep.Where(idx => idx != ci).ToList();
                var sous = new ModeleEmpirique(trial.Select(idx => m.Contextes[idx]).ToList(), m.Issues);
                if (SectionsGlobales(sous, limit: 1).Count == 0)
                {
                    keep = trial;
                    changed = true;
                }
            }
        }
        return keep.ToArray();
    }

    public static ResultatVerdict Verdict(ModeleEmpirique m)
    {
        var resultat = new ResultatVerdict();
        var rapport = Rapport(m);
        resultat.Rapport = rapport;

        var sections = SectionsGlobales(m);
        resultat.SectionsGlobales = sections.Count;
        bool satExhaustive = sections.Count > 0;

        // Les sections locales sont comptees meme lorsque le modele est
        // insatisfaisable : elles servent alors a caracteriser l'obstruction.
        var local = SectionsLocales(m, sections);
        resultat.SectionsLocales = local.nLocal;
        resultat.SansExtension = local.nSansExtension;

        bool? satSolveur = ResoudreParPropagation(m);
        resultat.SolveurDisponible = true;
        resultat.SolveurSatisfiable = satSolveur;
        resultat.StatutSolveur = satSolveur == true ? "SAT" : (satSolveur == false ? "INFEASIBLE" : "UNKNOWN");
        if (satSolveur.HasValue && satSolveur.Value != satExhaustive)
            throw new InvalidOperationException("desaccord solveur CSP / enumeration exhaustive : bug d'encodage, pas un resultat");
        resultat.Satisfiable = satExhaustive;

        if (rapport.Degenerescence != null)
        {
            resultat.Verdict = rapport.Degenerescence;
            resultat.Rationale = rapport.RaisonDegenerescence;
            return resultat;
        }

        if (!satExhaustive)
        {
            resultat.Verdict = "strongly_contextual";
            resultat.Rationale = "aucune assignation globale ne se restreint dans le support de tous les contextes : obstruction au recollement (cran KS)";
            resultat.ObstructionMinimale = ObstructionMinimale(m);
            return resultat;
        }

        if (local.nSansExtension > 0)
        {
            resultat.Verdict = "logically_contextual";
            resultat.Rationale = local.nSansExtension.ToString(CultureInfo.InvariantCulture) + " section(s) locale(s) observee(s) ne s'etendent a aucune section globale : obstruction partielle, localisee sur ces evenements";
        }
        else
        {
            resultat.Verdict = "non_contextual";
            resultat.Rationale = sections.Count.ToString(CultureInfo.InvariantCulture) + " section(s) globale(s), et toute section locale observee s'etend : le modele est non contextuel, l'analogie KS meurt proprement";
        }
        return resultat;
    }
}

public static class Fixtures
{
    public static ModeleEmpirique BoitePr()
    {
        int[][] xor0 = { new[] { 0, 0 }, new[] { 1, 1 } };
        int[][] xor1 = { new[] { 0, 1 }, new[] { 1, 0 } };
        return ModeleEmpirique.FromContexts(new[]
        {
            new Contexte(new[] { "a1", "b1" }, xor0),
            new Contexte(new[] { "a1", "b2" }, xor0),
            new Contexte(new[] { "a2", "b1" }, xor0),
            new Contexte(new[] { "a2", "b2" }, xor1),
        }, new[] { 0, 1 });
    }

    public static ModeleEmpirique LogiqueMaisPasForte()
    {
        int[][] xor0 = { new[] { 0, 0 }, new[] { 1, 1 } };
        int[][] elargi = { new[] { 0, 0 }, new[] { 0, 1 }, new[] { 1, 0 } };
        return ModeleEmpirique.FromContexts(new[]
        {
            new Contexte(new[] { "a1", "b1" }, xor0),
            new Contexte(new[] { "a1", "b2" }, xor0),
            new Contexte(new[] { "a2", "b1" }, xor0),
            new Contexte(new[] { "a2", "b2" }, elargi),
        }, new[] { 0, 1 });
    }

    public static ModeleEmpirique NonContextuel()
    {
        int[][] plein = { new[] { 0, 0 }, new[] { 0, 1 }, new[] { 1, 0 }, new[] { 1, 1 } };
        return ModeleEmpirique.FromContexts(new[]
        {
            new Contexte(new[] { "a1", "b1" }, plein),
            new Contexte(new[] { "a1", "b2" }, plein),
            new Contexte(new[] { "a2", "b1" }, plein),
            new Contexte(new[] { "a2", "b2" }, plein),
        }, new[] { 0, 1 });
    }
}

public static class Verifs
{
    public static void Entier(string nom, long calcule, long attendu)
    {
        bool ok = calcule == attendu;
        Console.WriteLine((ok ? "CONFORME" : "ECART") + " — " + nom + " : calcule = "
            + calcule.ToString(CultureInfo.InvariantCulture) + " | attendu = "
            + attendu.ToString(CultureInfo.InvariantCulture));
        if (!ok) throw new InvalidOperationException("Ecart sur le controle « " + nom + " »");
    }

    public static void Entier(string nom, BigInteger calcule, BigInteger attendu)
    {
        bool ok = calcule == attendu;
        Console.WriteLine((ok ? "CONFORME" : "ECART") + " — " + nom + " : calcule = "
            + calcule.ToString(CultureInfo.InvariantCulture) + " | attendu = "
            + attendu.ToString(CultureInfo.InvariantCulture));
        if (!ok) throw new InvalidOperationException("Ecart sur le controle « " + nom + " »");
    }

    public static void Vrai(string nom, bool condition)
    {
        Console.WriteLine((condition ? "CONFORME" : "ECART") + " — " + nom);
        if (!condition) throw new InvalidOperationException("Ecart sur le controle « " + nom + " »");
    }
}

var pr = Fixtures.BoitePr();

Console.WriteLine("contextes :");
foreach (var ctx in pr.Contextes)
    Console.WriteLine("  (" + string.Join(", ", ctx.Proxies.Select(p => "'" + p + "'")) + ")  support = ["
        + string.Join(", ", ctx.Support.Select(t => "(" + string.Join(", ", t.Select(v => v.ToString(CultureInfo.InvariantCulture))) + ")")) + "]");
Console.WriteLine();
Console.WriteLine("observables      : (" + string.Join(", ", pr.Mesures.Select(m => "'" + m + "'")) + ")");
Console.WriteLine("espace de recherche : " + pr.EspaceRecherche.ToString(CultureInfo.InvariantCulture));

var verdict = Csp.Verdict(pr);
Console.WriteLine();
Console.WriteLine("satisfiable : " + (verdict.Satisfiable == true ? "True" : "False"));
Console.WriteLine("VERDICT     : " + verdict.Verdict);
Console.WriteLine("solveur CSP : disponible = True | satisfiable = " + (verdict.SolveurSatisfiable == true ? "True" : "False")
    + " | statut = " + verdict.StatutSolveur);
Console.WriteLine("obstruction minimale : (" + string.Join(", ", verdict.ObstructionMinimale ?? Array.Empty<int>()) + ")");

Verifs.Entier("nombre de contextes de la boite PR", pr.Contextes.Count, 4);
Verifs.Entier("espace de recherche (2 issues ^ 4 observables)", pr.EspaceRecherche, BigInteger.Pow(2, 4));
Verifs.Entier("sections globales (enumeration exhaustive)", verdict.SectionsGlobales, 0);
Verifs.Vrai("les deux voies (enumeration exhaustive et solveur CSP) sont d'accord", verdict.SolveurSatisfiable == verdict.Satisfiable);

int nAffectations = 0;
bool gaucheToujoursPair = true;
foreach (var v1 in pr.Issues)
    foreach (var v2 in pr.Issues)
        foreach (var v3 in pr.Issues)
            foreach (var v4 in pr.Issues)
            {
                int a1 = v1, a2 = v2, b1 = v3, b2 = v4;
                int gauche = (a1 ^ b1) + (a1 ^ b2) + (a2 ^ b1) + (a2 ^ b2);
                if (gauche % 2 != 0) gaucheToujoursPair = false;
                nAffectations++;
            }
Verifs.Vrai("parite : le membre de gauche est pair sur les " + nAffectations.ToString(CultureInfo.InvariantCulture) + " affectations, le membre de droite est impair", gaucheToujoursPair);

var minimale = verdict.ObstructionMinimale ?? Array.Empty<int>();
bool tousRetraitsSatisfiables = minimale.All(ci =>
    Csp.SectionsGlobales(new ModeleEmpirique(pr.Contextes.Where((_, idx) => idx != ci).ToList(), pr.Issues), limit: 1).Count > 0);
Verifs.Entier("taille de l'obstruction minimale", minimale.Length, 4);
Verifs.Vrai("l'obstruction minimale est irreductible (chaque contexte retiré rend le modele satisfiable)", tousRetraitsSatisfiables);

contextes :


  ('a1', 'b1')  support = [(0, 0), (1, 1)]


  ('a1', 'b2')  support = [(0, 0), (1, 1)]


  ('a2', 'b1')  support = [(0, 0), (1, 1)]


  ('a2', 'b2')  support = [(0, 1), (1, 0)]


observables      : ('a1', 'a2', 'b1', 'b2')


... (8 outputs left out)

CONFORME — les deux voies (enumeration exhaustive et solveur CSP) sont d'accord


CONFORME — parite : le membre de gauche est pair sur les 16 affectations, le membre de droite est impair


CONFORME — taille de l'obstruction minimale : calcule = 4 | attendu = 4


CONFORME — l'obstruction minimale est irreductible (chaque contexte retiré rend le modele satisfiable)


### Lecture

Le détecteur retrouve `strongly_contextual` sur un cas dont nous connaissons la réponse
par un argument de parité, et les **deux voies indépendantes s'accordent** : l'énumération
exhaustive bornée et le solveur CSP (résolution par propagation). Un désaccord entre elles
lèverait une `InvalidOperationException` — ce serait un bug d'encodage, jamais un résultat.

Noter que le solveur CSP **ne rend aucun témoin**. Il répond seulement par un statut
`SAT`/`UNSAT`, calculé par un parcours déterministe sur le modèle fourni. Aucun témoin
non canonique n'est donc produit, et aucun choix arbitraire n'est introduit dans les
sorties committées.

L'**obstruction minimale** répond à une exigence de l'issue : en cas d'obstruction, dire
*quels protocoles* portent le conflit. C'est une sous-famille irréductible — en retirer
n'importe quel contexte rend le modèle satisfiable.

## 2. Trois crans, pas deux

Voici l'erreur qu'un test naïf commet. On construit le CSP, il répond `SAT`, on conclut
« non contextuel ». **C'est faux.** `SAT` dit seulement « pas *fortement* contextuel ».

Abramsky et Brandenburger distinguent un cran intermédiaire, la **contextualité logique** :
des sections globales existent, mais il existe un événement local **effectivement observé**
qu'aucune de ces sections globales ne réalise. L'obstruction n'est pas globale, elle est
*localisée sur cet événement* — et un test SAT nu ne la voit pas.

In [2]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Illustration 2 — les trois crans de la hierarchie de contextualite.
// Un `SAT` nu ne dit que « pas fortement contextuel » : le cran intermediaire
// (contextualite logique) se lit separement, section par section.
var modeles = new List<(string nom, ModeleEmpirique modele)>
{
    ("boite PR", Fixtures.BoitePr()),
    ("logique mais pas forte", Fixtures.LogiqueMaisPasForte()),
    ("non contextuel", Fixtures.NonContextuel()),
};

var verdicts = modeles
    .Select(e => (nom: e.nom, modele: e.modele, v: Csp.Verdict(e.modele)))
    .ToList();

Console.WriteLine("modele".PadRight(24) + "SAT".PadRight(7) + "verdict".PadRight(24) + "sections".PadRight(10) + "locales sans extension");
Console.WriteLine(new string('-', 92));
foreach (var item in verdicts)
    Console.WriteLine(item.nom.PadRight(24)
        + (item.v.Satisfiable == true ? "True" : "False").PadRight(7)
        + item.v.Verdict.PadRight(24)
        + item.v.SectionsGlobales.ToString(CultureInfo.InvariantCulture).PadRight(10)
        + item.v.SansExtension.ToString(CultureInfo.InvariantCulture) + "/"
        + item.v.SectionsLocales.ToString(CultureInfo.InvariantCulture));

// Controles calcules : les trois crans sont lus dans les valeurs produites.
Verifs.Entier("sections globales de la boite PR", verdicts[0].v.SectionsGlobales, 0);
Verifs.Entier("sections locales sans extension de la boite PR", verdicts[0].v.SansExtension, 8);
Verifs.Entier("sections globales du modele logiquement contextuel", verdicts[1].v.SectionsGlobales, 1);
Verifs.Entier("sections locales sans extension du modele logiquement contextuel", verdicts[1].v.SansExtension, 5);
Verifs.Entier("sections globales du modele plein (2 issues ^ 4 observables)", verdicts[2].v.SectionsGlobales, 16);
Verifs.Entier("sections locales sans extension du modele plein", verdicts[2].v.SansExtension, 0);
Verifs.Vrai("le modele logiquement contextuel est satisfiable mais porte des temoins",
    verdicts[1].v.Satisfiable == true && verdicts[1].v.SansExtension > 0 && verdicts[1].v.Verdict == "logically_contextual");
Verifs.Vrai("le modele plein n'est pas contextuel", verdicts[2].v.Verdict == "non_contextual");

modele                  SAT    verdict                 sections  locales sans extension


--------------------------------------------------------------------------------------------


boite PR                False  strongly_contextual     0         8/8


logique mais pas forte  True   logically_contextual    1         5/9


non contextuel          True   non_contextual          16        0/16


CONFORME — sections globales de la boite PR : calcule = 0 | attendu = 0


... (3 outputs left out)

CONFORME — sections globales du modele plein (2 issues ^ 4 observables) : calcule = 16 | attendu = 16


CONFORME — sections locales sans extension du modele plein : calcule = 0 | attendu = 0


CONFORME — le modele logiquement contextuel est satisfiable mais porte des temoins


CONFORME — le modele plein n'est pas contextuel


### Lecture

La ligne du milieu est celle qui compte. Le modèle est **satisfiable** — un test naïf
aurait conclu « non contextuel » — et pourtant il porte des témoins : des événements
observés localement qu'aucune section globale n'étend. Le verdict correct est
`logically_contextual`.

C'est la raison pour laquelle le module rend `logical_contextuality` séparément : sans ce
cran, la moitié des obstructions passent pour des absences d'obstruction.

### Exercice 1 — construire un modèle et prédire son cran

Le recouvrement **partiel** est ce qui rend la question non triviale : trois contextes
$A=\{p,q\}$, $B=\{q,r\}$, $C=\{r,p\}$ se chevauchent deux à deux sans qu'aucun ne
contienne les trois proxys. C'est la structure en cycle qui porte les obstructions.

Construis un tel modèle avec `ModeleEmpirique.FromContexts`, **prédis** son cran avant de lancer le
test, puis compare. Le format attendu par `FromContexts` est une liste de contextes
`new Contexte(new[] { "p", "q" }, new[] { new[] { 0, 0 }, new[] { 0, 1 } })`.

In [3]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Exercice 1 — construire un modele a recouvrement partiel en cycle.
//
// TODO etudiant : construire un modele a recouvrement partiel en cycle.
//
// Indice 1 : trois contextes A={p,q}, B={q,r}, C={r,p}, issues dans {0,1}.
// Indice 2 : pour forcer une obstruction, demande a chaque contexte que ses deux
//            proxys DIFFERENT (support = [(0,1),(1,0)]). Un cycle impair de
//            contraintes « different » est insatisfiable -- c'est le meme argument
//            de parite que la boite PR.
// Indice 3 : pour obtenir un modele satisfiable, remplace l'un des trois supports
//            par [(0,0),(1,1)] (« egaux ») et refais le test.
//
// Etape 1 : ecrire les trois contextes
List<Contexte>? contextes = null;

// Etape 2 : construire le modele
// var modele = ModeleEmpirique.FromContexts(contextes!, new[] { 0, 1 });

// Etape 3 : predire le verdict AVANT de le calculer, puis verifier
string? prediction = null;

Console.WriteLine("Exercice 1 a completer");

Exercice 1 a completer


> **Encadré — le théorème derrière l'exercice.** L'argument de parité de l'exercice (un cycle impair de contraintes « différent » est insatisfiable) est un théorème nommé : un graphe est **biparti** si et seulement s'il ne contient **aucun cycle impair** (König). Assigner une valeur booléenne à chaque proxy en respectant « adjacent ⟹ différent » sur le graphe de chevauchement des contextes, c'est exactement une **2-coloration** de ce graphe — elle existe précisément quand le graphe est biparti. Le triangle `A={p,q}, B={q,r}, C={r,p}` est le cycle impair minimal de la littérature, et c'est pourquoi il sert de cas d'école.
>
> Ce que le théorème **ne** dit pas, et que votre propre exécution doit établir : le cran exact auquel votre construction cesse d'être satisfiable, et si une relaxation « égal » sur une seule arête suffit à restaurer une 2-coloration. L'encadré n'anticipe pas ces deux réponses — il nomme le critère de parité qui les rend calculables. C'est l'exemple type du principe « généraliser allège » (S. Sheydvasser, *The Power of Abstraction*, 26/09/2026 — distillation #19333, volet A.2).

## 3. Le garde-fou : trois manières d'être vide

C'est le point méthodologique central de cette annexe, et il vaut bien au-delà d'elle.

L'issue #7290 nomme le risque : *« le risque principal est l'analogie décorative »* — un
appareil impressionnant qui rend un verdict dont la réponse était **forcée par la
construction**. La série ICT en a déjà fait l'expérience avec l'obstruction de Čech
(ICT-15d) : verdict `TRIVIAL`, parce que les sections étaient colinéaires par construction.

La leçon est convertie ici en machinerie plutôt qu'en vigilance : **un verdict ne vaut que
si sa négation était atteignable**. Trois configurations reçoivent donc un verdict *distinct*
de `SAT`/`UNSAT`, qui dit « le test n'était pas capable de trancher » :

| Verdict de dégénérescence | Ce qui est vide |
|---|---|
| `degenerate_single_cover` | Tous les contextes co-mesurent **le même** ensemble de proxys. La question se réduit à « l'intersection des supports est-elle non vide ? ». Il n'y a pas de contextes distincts, donc pas de contextualité possible. |
| `degenerate_no_overlap` | Aucun proxy n'est partagé. `SAT` est forcé : chaque contexte se satisfait indépendamment. |
| `degenerate_rigid` | Tous les supports sont des singletons. Un `UNSAT` n'est alors qu'un **changement de valeur** — une dissociation, pas une obstruction. La force de Kochen–Specker est qu'aucune attribution ne marche *alors que chaque contexte en admet localement plusieurs*. |

L'ordre de priorité est `single_cover` → `no_overlap` → `rigid` : on retient la
dégénérescence la plus forte, celle qui force le verdict de la façon la plus complète.

In [4]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Illustration 3 — les trois manieres d'etre vide (le garde-fou).
//
// Un verdict ne vaut que si sa negation etait atteignable : chacune de ces
// trois configurations recoit un verdict distinct de SAT/UNSAT, qui dit
// « le test n'etait pas capable de trancher ».
var demos = new List<(string nom, List<Contexte> contextes)>
{
    ("recouvrement unique", new List<Contexte>
    {
        new Contexte(new[] { "p", "q" }, new[] { new[] { 0, 0 }, new[] { 0, 1 } }),
        new Contexte(new[] { "p", "q" }, new[] { new[] { 0, 0 }, new[] { 1, 1 } }),
    }),
    ("aucun recouvrement", new List<Contexte>
    {
        new Contexte(new[] { "p" }, new[] { new[] { 0 }, new[] { 1 } }),
        new Contexte(new[] { "q" }, new[] { new[] { 0 }, new[] { 1 } }),
    }),
    ("supports rigides", new List<Contexte>
    {
        new Contexte(new[] { "p", "q" }, new[] { new[] { 0, 1 } }),
        new Contexte(new[] { "q", "r" }, new[] { new[] { 0, 1 } }),
    }),
};

Console.WriteLine("construction".PadRight(24) + "SAT".PadRight(7) + "verdict");
Console.WriteLine(new string('-', 62));
foreach (var demo in demos)
{
    var modele = ModeleEmpirique.FromContexts(demo.contextes);
    var v = Csp.Verdict(modele);
    Console.WriteLine(demo.nom.PadRight(24)
        + (v.Satisfiable == true ? "True" : "False").PadRight(7)
        + v.Verdict);
}

// Controles calcules : chaque verdict est celui PRODUIT par le detecteur, et on
// le compare aux valeurs attendues de la construction (jamais une loi annoncee
// sans recalcul).
var verdicts = demos.ToDictionary(
    d => d.nom,
    d => Csp.Verdict(ModeleEmpirique.FromContexts(d.contextes)),
    StringComparer.Ordinal);

Verifs.Vrai("recouvrement unique : verdict degenerate_single_cover",
    verdicts["recouvrement unique"].Verdict == "degenerate_single_cover");
Verifs.Vrai("recouvrement unique : le modele reste satisfiable",
    verdicts["recouvrement unique"].Satisfiable == true);
Verifs.Vrai("aucun recouvrement : verdict degenerate_no_overlap",
    verdicts["aucun recouvrement"].Verdict == "degenerate_no_overlap");
Verifs.Vrai("aucun recouvrement : le modele reste satisfiable",
    verdicts["aucun recouvrement"].Satisfiable == true);
Verifs.Vrai("supports rigides : verdict degenerate_rigid",
    verdicts["supports rigides"].Verdict == "degenerate_rigid");
Verifs.Vrai("supports rigides : le modele est insatisfaisable",
    verdicts["supports rigides"].Satisfiable == false);
Verifs.Vrai("supports rigides : le verdict n'est JAMAIS strongly_contextual",
    verdicts["supports rigides"].Verdict != "strongly_contextual");

// Le support rigide force bien le « changement de valeur » : le contexte 1
// impose q = 1, le contexte 2 impose q = 0 (constante recalculee, pas ecrite
// en dur dans l'affirmation). Dans le contexte 2, q est la premiere colonne du
// tuple (les proxys sont {q,r}) ; dans le contexte 1, q est la seconde
// colonne (les proxys sont {p,q}).
var modeleRigide = ModeleEmpirique.FromContexts(demos[2].contextes);
int qContexte1 = modeleRigide.Contextes[0].Support[0][1];
int qContexte2 = modeleRigide.Contextes[1].Support[0][0];
Verifs.Entier("valeur de q imposee par le contexte 1", qContexte1, 1);
Verifs.Entier("valeur de q imposee par le contexte 2", qContexte2, 0);

construction            SAT    verdict


--------------------------------------------------------------


recouvrement unique     True   degenerate_single_cover


aucun recouvrement      True   degenerate_no_overlap


supports rigides        False  degenerate_rigid


CONFORME — recouvrement unique : verdict degenerate_single_cover


... (4 outputs left out)

CONFORME — supports rigides : le modele est insatisfaisable


CONFORME — supports rigides : le verdict n'est JAMAIS strongly_contextual


CONFORME — valeur de q imposee par le contexte 1 : calcule = 1 | attendu = 1


CONFORME — valeur de q imposee par le contexte 2 : calcule = 0 | attendu = 0


### Lecture — la ligne décisive est la troisième

« supports rigides » est **insatisfiable** : le contexte 1 impose `q = 1`, le contexte 2
impose `q = 0`. Un test naïf aurait rapporté « obstruction structurelle », et cela aurait
été un **faux positif** — pas une erreur de calcul, mais une erreur d'interprétation. Il
n'y a là aucune obstruction au sens de Kochen–Specker : simplement deux protocoles qui ne
mesurent pas la même chose. Chaque contexte n'admettant qu'une seule section locale, il
n'y avait aucune liberté à contraindre.

C'est exactement le faux positif que l'issue redoutait, et le garde l'intercepte.

### Exercice 2 — falsifier le garde

Un garde-fou qu'on n'a pas essayé de casser n'est pas un garde-fou. Écris un modèle qui
soit **à la fois** insatisfiable **et** de supports rigides, mais avec un recouvrement
*réel* (au moins un proxy partagé, et pas le même ensemble de proxys partout).

Vérifie que le verdict rendu est bien `degenerate_rigid` et **jamais**
`strongly_contextual`. Si tu parviens à obtenir `strongly_contextual` sur un modèle à
supports tous singletons, tu as trouvé un bug : signale-le sur l'issue #7290.

In [5]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Exercice 2 — fabriquer un piege pour le garde-fou.
//
// TODO etudiant : fabriquer un piege pour le garde-fou.
//
// Indice 1 : recouvrement REEL veut dire que les ensembles de proxys ne sont pas
//            tous identiques (sinon c'est single_cover, une autre degenerescence,
//            et elle a la priorite).
// Indice 2 : supports rigides = chaque support ne contient qu'un seul tuple.
// Indice 3 : le test decisif du module s'appelle
//            test_rigid_supports_unsat_is_not_reported_as_contextual --
//            tu es en train d'en ecrire une variante.
//
// Etape 1 : construire le modele piege
List<Contexte>? piege = null;

// Etape 2 : verifier le verdict et l'assertion attendue
// var v = Csp.Verdict(ModeleEmpirique.FromContexts(piege!));
// attendu : v.Satisfiable == false et v.Verdict == "degenerate_rigid"

Console.WriteLine("Exercice 2 a completer");

Exercice 2 a completer


## 4. La mesure appliquée : le zoo ICT

Tout est en place. On braque maintenant l'instrument validé sur des données réelles.

**Protocole.** Substrat S1 (`TableauTriAutomatique`), trajectoire du désordre mesurée par le
nombre d'inversions. Contextes = trois grains de description ($k \in \{3,4,6\}$ symboles).
Graines $\{0, 1, 7, 42, 99\}$. Les proxys sont câblés **comme dans ICT-15c** — avec le
correctif anti-saturation $f(x) = x$ — plutôt que recâblés à neuf, pour que la mesure porte
sur le zoo tel qu'il est et non sur une variante de circonstance.

Les valeurs continues sont discrétisées en issues binaires par **médiane du corpus** :
c'est un **choix**, pas une donnée, et le verdict est à lire avec cette réserve.

In [6]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Illustration 4 — les signatures du zoo de proxys (substrat S1).
//
// Le generateur aleatoire est celui de Python (MT19937 + init_by_array +
// _randbelow) : a graine identique, la trajectoire est identique au jumeau
// Python, ce qui rend la comparaison des signatures legitimate.

public sealed class GenerateurPython
{
    private readonly uint[] _mt = new uint[624];
    private int _index;

    public GenerateurPython(int seed)
    {
        var key = new List<uint>();
        ulong n = seed < 0 ? (ulong)(-(long)seed) : (ulong)seed;
        if (n == 0)
        {
            key.Add(0u);
        }
        else
        {
            while (n > 0)
            {
                key.Add((uint)(n & 0xFFFFFFFFUL));
                n >>= 32;
            }
        }
        InitParTableau(key.ToArray());
    }

    private void InitParTableau(uint[] key)
    {
        _mt[0] = 19650218u;
        for (int idxInitialisation = 1; idxInitialisation < 624; idxInitialisation++)
        {
            _mt[idxInitialisation] = (uint)(1812433253u * (_mt[idxInitialisation - 1] ^ (_mt[idxInitialisation - 1] >> 30)) + (uint)idxInitialisation);
        }

        int indexMt = 1;
        int indexKey = 0;
        int k = Math.Max(624, key.Length);
        for (; k > 0; k--)
        {
            _mt[indexMt] = (uint)((_mt[indexMt] ^ ((_mt[indexMt - 1] ^ (_mt[indexMt - 1] >> 30)) * 1664525u)) + key[indexKey] + (uint)indexKey);
            indexMt++;
            indexKey++;
            if (indexMt >= 624)
            {
                _mt[0] = _mt[623];
                indexMt = 1;
            }
            if (indexKey >= key.Length)
            {
                indexKey = 0;
            }
        }

        for (k = 623; k > 0; k--)
        {
            _mt[indexMt] = (uint)((_mt[indexMt] ^ ((_mt[indexMt - 1] ^ (_mt[indexMt - 1] >> 30)) * 1566083941u)) - (uint)indexMt);
            indexMt++;
            if (indexMt >= 624)
            {
                _mt[0] = _mt[623];
                indexMt = 1;
            }
        }

        _mt[0] = 0x80000000u;
        _index = 624;
    }

    private uint NextUInt32()
    {
        if (_index >= 624)
        {
            for (int k = 0; k < 624; k++)
            {
                uint y = (_mt[k] & 0x80000000u) | (_mt[(k + 1) % 624] & 0x7FFFFFFFu);
                _mt[k] = _mt[(k + 397) % 624] ^ (y >> 1);
                if ((y & 1u) != 0u)
                {
                    _mt[k] ^= 2567483615u;
                }
            }
            _index = 0;
        }

        uint x = _mt[_index++];
        x ^= x >> 11;
        x ^= (x << 7) & 2636928640u;
        x ^= (x << 15) & 4022730752u;
        x ^= x >> 18;
        return x;
    }

    private uint GetRandBits(int k)
    {
        return NextUInt32() >> (32 - k);
    }

    private static int BitsNombre(int n)
    {
        int bits = 0;
        uint v = (uint)n;
        while (v > 0)
        {
            v >>= 1;
            bits++;
        }
        return bits;
    }

    // _randbelow(n) : tirage par rejection des valeurs hors intervalle.
    public int RandBelow(int n)
    {
        if (n <= 0)
        {
            return 0;
        }

        int k = BitsNombre(n);
        uint r = GetRandBits(k);
        while (r >= (uint)n)
        {
            r = GetRandBits(k);
        }
        return (int)r;
    }
}

public sealed class CelluleTri
{
    public int Valeur { get; set; }
    public string Algotype { get; set; } = "bubble";
    public bool Gelee { get; set; }
    public int Cid { get; set; }
}

public sealed class TableauTriAutomatique
{
    public List<CelluleTri> Cellules { get; } = new List<CelluleTri>();
    public GenerateurPython Alea { get; }
    public int Etapes { get; private set; }

    public TableauTriAutomatique(int[] valeurs, int graine)
    {
        Alea = new GenerateurPython(graine);
        for (int idx = 0; idx < valeurs.Length; idx++)
        {
            Cellules.Add(new CelluleTri
            {
                Valeur = valeurs[idx],
                Algotype = "bubble",
                Gelee = false,
                Cid = idx
            });
        }
    }

    public int[] Valeurs => Cellules.Select(c => c.Valeur).ToArray();

    private void Echanger(int i, int j)
    {
        CelluleTri tmp = Cellules[i];
        Cellules[i] = Cellules[j];
        Cellules[j] = tmp;
    }

    private int? VoisinCible(int i)
    {
        CelluleTri c = Cellules[i];
        if (c.Gelee)
        {
            return null;
        }

        if (c.Algotype == "bubble")
        {
            int j = i + 1;
            if (j < Cellules.Count && c.Valeur > Cellules[j].Valeur)
            {
                return j;
            }
        }
        else
        {
            int j = i - 1;
            if (j >= 0 && c.Valeur < Cellules[j].Valeur)
            {
                return j;
            }
        }
        return null;
    }

    // Un pas active une cellule non gelee tiree au hasard ; step() rend False
    // pour un tick sans mouvement, PAS pour une terminaison :
    // on ne coupe donc pas sur False.
    public bool Step()
    {
        var movable = Enumerable.Range(0, Cellules.Count)
            .Where(idx => !Cellules[idx].Gelee)
            .ToList();
        if (movable.Count == 0)
        {
            return false;
        }

        int i = movable[Alea.RandBelow(movable.Count)];
        int? j = VoisinCible(i);
        bool agi = false;
        if (j.HasValue)
        {
            Echanger(i, j.Value);
            agi = true;
        }
        Etapes++;
        return agi;
    }
}

public static class Banc
{
    public static int Inversions(int[] v)
    {
        int c = 0;
        for (int i = 0; i < v.Length; i++)
        {
            for (int j = i + 1; j < v.Length; j++)
            {
                if (v[i] > v[j])
                {
                    c++;
                }
            }
        }
        return c;
    }

    // Trajectoire du desordre du substrat S1.
    public static int[] Trajectoire(int graine, int nPas = 120)
    {
        var arr = new TableauTriAutomatique(
            new[] { 5, 3, 8, 1, 9, 2, 7, 4, 6, 0 },
            graine);
        var traj = new List<int> { Inversions(arr.Valeurs) };

        // step() rend False pour un tick sans mouvement, PAS pour une terminaison :
        // on ne coupe donc pas sur False.
        for (int k = 0; k < nPas; k++)
        {
            arr.Step();
            traj.Add(Inversions(arr.Valeurs));
        }
        return traj.ToArray();
    }

    // Coarse-graining : bins de largeur egale sur l'amplitude observee.
    public static int[] Discretiser(int[] traj, int nSymboles)
    {
        int lo = traj.Min();
        int hi = traj.Max();
        if (hi == lo)
        {
            return Enumerable.Repeat(0, traj.Length).ToArray();
        }
        return traj
            .Select(x => Math.Min(
                nSymboles - 1,
                (int)((x - lo) / (double)(hi - lo) * nSymboles)))
            .ToArray();
    }
}

public static class Spectral
{
    // P[i, j] = P(X_{t+1} = j | X_t = i), estimee par comptage, avec un
    // plancher de Laplace (evite les lignes nulles).
    public static double[,] MatriceTransition(int[] etats, int nSymboles, double laplace = 1e-9)
    {
        var p = new double[nSymboles, nSymboles];
        for (int i = 0; i < nSymboles; i++)
        {
            for (int j = 0; j < nSymboles; j++)
            {
                p[i, j] = laplace;
            }
        }

        for (int t = 0; t + 1 < etats.Length; t++)
        {
            int s = etats[t];
            int u = etats[t + 1];
            if (s >= 0 && s < nSymboles && u >= 0 && u < nSymboles)
            {
                p[s, u] += 1.0;
            }
        }

        for (int i = 0; i < nSymboles; i++)
        {
            double somme = 0;
            for (int j = 0; j < nSymboles; j++)
            {
                somme += p[i, j];
            }
            for (int j = 0; j < nSymboles; j++)
            {
                p[i, j] /= somme;
            }
        }
        return p;
    }

    // W = (P + P^T) / 2, diagonale nulle.
    public static double[,] GrapheSymetrise(double[,] p)
    {
        int n = p.GetLength(0);
        var w = new double[n, n];
        for (int i = 0; i < n; i++)
        {
            for (int j = 0; j < n; j++)
            {
                w[i, j] = (p[i, j] + p[j, i]) / 2.0;
            }
        }
        for (int i = 0; i < n; i++)
        {
            w[i, i] = 0.0;
        }
        return w;
    }

    // Valeurs propres d'une matrice symetrique (rotations de Jacobi).
    public static double[] ValeursPropresSymetriques(double[,] a, int n)
    {
        var m = (double[,])a.Clone();
        for (int balayage = 0; balayage < 200; balayage++)
        {
            double norme = 0;
            for (int p = 0; p < n; p++)
            {
                for (int q = p + 1; q < n; q++)
                {
                    norme += m[p, q] * m[p, q];
                }
            }
            if (norme < 1e-28)
            {
                break;
            }

            for (int p = 0; p < n - 1; p++)
            {
                for (int q = p + 1; q < n; q++)
                {
                    if (Math.Abs(m[p, q]) < 1e-300)
                    {
                        continue;
                    }
                    double theta = (m[q, q] - m[p, p]) / (2.0 * m[p, q]);
                    double t = (theta >= 0 ? 1.0 : -1.0)
                        / (Math.Abs(theta) + Math.Sqrt(theta * theta + 1.0));
                    double c = 1.0 / Math.Sqrt(t * t + 1.0);
                    double s = t * c;

                    for (int k = 0; k < n; k++)
                    {
                        double mkp = m[k, p];
                        double mkq = m[k, q];
                        m[k, p] = c * mkp - s * mkq;
                        m[k, q] = s * mkp + c * mkq;
                    }
                    for (int k = 0; k < n; k++)
                    {
                        double mpk = m[p, k];
                        double mqk = m[q, k];
                        m[p, k] = c * mpk - s * mqk;
                        m[q, k] = s * mpk + c * mqk;
                    }
                }
            }
        }

        var vals = new double[n];
        for (int i = 0; i < n; i++)
        {
            vals[i] = m[i, i];
        }
        Array.Sort(vals);
        return vals;
    }

    // Gap spectral du Laplacien symetrique : lambda_2 - lambda_1.
    public static double GapSpectral(int[] etats, int nSymboles)
    {
        var w = GrapheSymetrise(MatriceTransition(etats, nSymboles));
        int n = nSymboles;
        var l = new double[n, n];
        for (int i = 0; i < n; i++)
        {
            double degre = 0;
            for (int j = 0; j < n; j++)
            {
                degre += w[i, j];
            }
            l[i, i] = degre;
            for (int j = 0; j < n; j++)
            {
                if (i != j)
                {
                    l[i, j] = -w[i, j];
                }
            }
        }
        var eigs = ValeursPropresSymetriques(l, n);
        return eigs.Length < 2 ? double.NaN : eigs[1] - eigs[0];
    }
}

public static class Sensibilite
{
    // Encodage par premiere apparition : les ids des noeuds visites occupent
    // exactement 0..n_visite-1, les ids restants sont les noeuds jamais visites.
    public static int[] Encoder(int[] etats, int nSymboles)
    {
        var map = new Dictionary<int, int>();
        var ids = new int[etats.Length];
        for (int i = 0; i < etats.Length; i++)
        {
            if (!map.ContainsKey(etats[i]))
            {
                if (map.Count >= nSymboles)
                {
                    throw new ArgumentException("Plus d'etats distincts que n_symbols");
                }
                map[etats[i]] = map.Count;
            }
            ids[i] = map[etats[i]];
        }
        return ids;
    }

    // Adjacence STRUCTURELLE : la transition a-t-elle ete observee ? (boucles exclues)
    public static bool[,] AdjacenceObservee(int[] etats, int nSymboles)
    {
        var a = new bool[nSymboles, nSymboles];
        for (int t = 0; t + 1 < etats.Length; t++)
        {
            int s = etats[t];
            int u = etats[t + 1];
            if (s == u)
            {
                continue;
            }
            if (s >= 0 && s < nSymboles && u >= 0 && u < nSymboles)
            {
                a[s, u] = true;
                a[u, s] = true;
            }
        }
        return a;
    }

    // f(x) = x (correctif anti-saturation) : la sensibilite locale est le
    // degre structurel observe, sur les noeuds effectivement visites.
    public static (double max, double mean, int nVisites) Distribution(int[] etats, int nSymboles)
    {
        int[] ids = Encoder(etats, nSymboles);
        var a = AdjacenceObservee(ids, nSymboles);
        var s = new int[nSymboles];
        for (int x = 0; x < nSymboles; x++)
        {
            int c = 0;
            for (int y = 0; y < nSymboles; y++)
            {
                if (a[x, y] && y != x)
                {
                    c++;
                }
            }
            s[x] = c;
        }

        int nVisite = ids.Distinct().Count();
        var visites = s.Take(nVisite).ToArray();
        double max = visites.Length == 0 ? 0.0 : visites.Max();
        double mean = visites.Length == 0 ? 0.0 : visites.Average();
        return (max, mean, nVisite);
    }
}

var SEEDS = new[] { 0, 1, 7, 42, 99 };
var GRAININGS = new[] { 3, 4, 6 };
var PROXIES = new[] { "spectral_gap", "sensitivity_mean", "sensitivity_max" };

// Les trois proxys partagent la meme signature d'appel : ce sont les etats
// discretises et le nombre de symboles. Rien n'empêche de les calculer tous
// les trois sur la meme trajectoire.
var signatures = new Dictionary<string, List<Dictionary<string, double>>>(StringComparer.Ordinal);
foreach (var k in GRAININGS)
{
    var runs = new List<Dictionary<string, double>>();
    foreach (var graine in SEEDS)
    {
        var states = Banc.Discretiser(Banc.Trajectoire(graine), k);
        var gap = Spectral.GapSpectral(states, k);
        var sens = Sensibilite.Distribution(states, k);
        runs.Add(new Dictionary<string, double>(StringComparer.Ordinal)
        {
            ["spectral_gap"] = gap,
            ["sensitivity_mean"] = sens.mean,
            ["sensitivity_max"] = sens.max,
        });
    }
    signatures["coarse_graining_k" + k.ToString(CultureInfo.InvariantCulture)] = runs;
}

foreach (var kv in signatures)
{
    Console.WriteLine(kv.Key);
    for (int i = 0; i < kv.Value.Count; i++)
    {
        var r = kv.Value[i];
        Console.WriteLine(
            "  seed=" + SEEDS[i].ToString(CultureInfo.InvariantCulture).PadLeft(2)
            + "  gap=" + r["spectral_gap"].ToString("F4", CultureInfo.InvariantCulture)
            + "  sens_mean=" + r["sensitivity_mean"].ToString("F4", CultureInfo.InvariantCulture)
            + "  sens_max=" + r["sensitivity_max"].ToString("F4", CultureInfo.InvariantCulture));
    }
}

// Controles calcules sur les grandeurs produites (jamais des assertions en dur).
Verifs.Entier("longueur d'une trajectoire (etat initial + 120 pas)", Banc.Trajectoire(0).Length, 121);
Verifs.Entier("nombre de contextes (grains de description)", signatures.Count, 3);
Verifs.Entier("nombre de graines par contexte", signatures["coarse_graining_k3"].Count, 5);
Verifs.Entier("nombre de proxys par signature", signatures["coarse_graining_k6"][0].Count, 3);

coarse_graining_k3


  seed= 0  gap=0.0193  sens_mean=1.3333  sens_max=2.0000


  seed= 1  gap=0.0195  sens_mean=1.3333  sens_max=2.0000


  seed= 7  gap=0.0135  sens_mean=1.3333  sens_max=2.0000


  seed=42  gap=0.0230  sens_mean=1.3333  sens_max=2.0000


  seed=99  gap=0.0238  sens_mean=1.3333  sens_max=2.0000


... (12 outputs left out)

CONFORME — longueur d'une trajectoire (etat initial + 120 pas) : calcule = 121 | attendu = 121


CONFORME — nombre de contextes (grains de description) : calcule = 3 | attendu = 3


CONFORME — nombre de graines par contexte : calcule = 5 | attendu = 5


CONFORME — nombre de proxys par signature : calcule = 3 | attendu = 3


In [7]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Illustration 5 — du modele empirique au verdict : seuils, diagnostic de
// recouvrement, satisfiabilite et verdict final du zoo de proxys.

var proxies = new[] { "spectral_gap", "sensitivity_mean", "sensitivity_max" };

// Seuils de discretisation : mediane de chaque proxy, toutes mesures
// confondues. La discretisation est un CHOIX, pas une donnee.
var seuils = new Dictionary<string, double>(StringComparer.Ordinal);
foreach (var p in proxies)
{
    var vals = signatures.Values
        .SelectMany(runs => runs)
        .Select(r => r[p])
        .OrderBy(v => v)
        .ToArray();
    int n = vals.Length;
    seuils[p] = n % 2 == 1
        ? vals[n / 2]
        : 0.5 * (vals[n / 2 - 1] + vals[n / 2]);
}

// Un contexte = une signature par graine ; le support est l'ensemble des
// tuples d'issues EFFECTIVLEMENT observes.
var contextes = new List<Contexte>();
foreach (var kv in signatures)
{
    var tuples = new List<int[]>();
    foreach (var run in kv.Value)
    {
        var tuple = proxies
            .Select(p => run[p] >= seuils[p] ? 1 : 0)
            .ToArray();
        if (!tuples.Any(existing => existing.SequenceEqual(tuple)))
        {
            tuples.Add(tuple);
        }
    }

    tuples.Sort((x, y) =>
    {
        for (int i = 0; i < Math.Min(x.Length, y.Length); i++)
        {
            int comparaison = x[i].CompareTo(y[i]);
            if (comparaison != 0)
            {
                return comparaison;
            }
        }
        return x.Length.CompareTo(y.Length);
    });

    contextes.Add(new Contexte(proxies, tuples.ToArray()));
}

var modele = ModeleEmpirique.FromContexts(contextes, new[] { 0, 1 });
var rapport = Csp.Rapport(modele);
var resultat = Csp.Verdict(modele);

Console.WriteLine("Seuils de discretisation (medianes du corpus)");
foreach (var p in proxies)
{
    Console.WriteLine(
        "  " + p.PadRight(18)
        + seuils[p].ToString("F6", CultureInfo.InvariantCulture));
}

Console.WriteLine();
Console.WriteLine("Diagnostic de recouvrement");
Console.WriteLine(
    "  contextes                : "
    + modele.Contextes.Count.ToString(CultureInfo.InvariantCulture));
Console.WriteLine(
    "  recouvrements distincts  : "
    + rapport.NRecouvrementsDistincts.ToString(CultureInfo.InvariantCulture));
Console.WriteLine(
    "  proxys partages          : ["
    + string.Join(", ", rapport.ProxysPartages.Select(p => "'" + p + "'"))
    + "]");
Console.WriteLine(
    "  tailles de support       : ["
    + string.Join(
 ", ",
        rapport.TaillesSupport.Select(s => s.ToString(CultureInfo.InvariantCulture)))
    + "]");

Console.WriteLine();
Console.WriteLine(
    "  satisfiable              : "
    + (resultat.Satisfiable == true ? "True" : "False"));
Console.WriteLine(
    "  solveur CSP              : disponible = True | satisfiable = "
    + (resultat.SolveurSatisfiable == true ? "True" : "False")
    + " | statut = " + resultat.StatutSolveur);
Console.WriteLine("  VERDICT                  : " + resultat.Verdict);

// Controles calcules : le recouvrement est lu AVANT le verdict, et les
// valeurs annoncees sont celles produites par le detecteur.
Verifs.Entier(
    "nombre de contextes",
    modele.Contextes.Count,
    3);
Verifs.Entier(
    "recouvrements distincts",
    rapport.NRecouvrementsDistincts,
    1);
Verifs.Entier(
    "nombre de proxys partages",
    rapport.ProxysPartages.Count,
    3);
Verifs.Vrai(
    "tailles de support = [1, 2, 1]",
    string.Join(
        ", ",
        rapport.TaillesSupport.Select(s => s.ToString(CultureInfo.InvariantCulture))) == "1, 2, 1");
Verifs.Vrai(
    "le modele est insatisfaisable",
    resultat.Satisfiable == false);
Verifs.Vrai(
    "le verdict est degenerate_single_cover",
    resultat.Verdict == "degenerate_single_cover");
Verifs.Vrai(
    "les deux voies (enumeration exhaustive et solveur CSP) sont d'accord",
    resultat.SolveurSatisfiable == resultat.Satisfiable);

Seuils de discretisation (medianes du corpus)


  spectral_gap      0.013437


  sensitivity_mean  1.500000


  sensitivity_max   2.000000


Diagnostic de recouvrement


  contextes                : 3


... (9 outputs left out)

CONFORME — tailles de support = [1, 2, 1]


CONFORME — le modele est insatisfaisable


CONFORME — le verdict est degenerate_single_cover


CONFORME — les deux voies (enumeration exhaustive et solveur CSP) sont d'accord


### Lecture — l'écart entre les deux dernières lignes **est** le résultat

Le modèle est **insatisfiable**, et le verdict n'est **pas** « fortement contextuel ». Cet
écart est tout le propos de l'annexe.

Un test naïf aurait rapporté ici « obstruction structurelle dans le zoo de proxys ». Le
garde intercepte, et la raison est structurelle — vérifiable sans le solveur : les trois
proxys du zoo partagent la même signature d'appel `(states, n_symbols)`. **Rien n'empêche
donc de les calculer tous les trois sur la même trajectoire.** Or c'est précisément la
*non*-co-mesurabilité — l'analogue de la non-commutativité chez Kochen–Specker — qui crée
des contextes distincts. Il n'y a qu'un seul recouvrement : `degenerate_single_cover`.

L'`UNSAT` observé n'est donc qu'un désaccord entre grains de description : un fait déjà
acquis par ailleurs dans la série, et d'une autre nature.

### Le verdict honnête

Le zoo ICT n'est **pas** « non contextuel ». La question de sa contextualité **n'y est pas
encore posable** — le transfert échoue un cran *avant* le `SAT`/`UNSAT`, faute de proxys
non conjointement mesurables.

L'ingrédient manquant est identifié, et il est unique : une structure de co-mesurabilité
**partielle et justifiée** — deux proxys dont les prétraitements sont *genuinement*
incompatibles, tels qu'aucun protocole ne les livre ensemble. L'outillage est livré validé :
le jour où une telle structure existe, le test se pose sans réécriture.

C'est un résultat négatif, et il est publiable tel quel. Ce qu'il ferme est une piste
séduisante ; ce qu'il laisse est un critère net pour la rouvrir.

### Un constat secondaire, rapporté à part

Les sorties ci-dessus montrent la saturation de l'instrument **malgré** le correctif
$f(x) = x$ d'ICT-15c : `sensitivity_max` vaut **2,0 sur les 5 graines comme sur les
3 grains** — il ne suit pas $k - 1$, qui vaudrait 3 à $k = 4$ et 5 à $k = 6$ — et
`sensitivity_mean` est constant par grain (1,33 ; 1,50 ; 1,67), identique d'une graine
à l'autre. Deux des trois proxys ne portent donc aucune variance inter-graines — seul
l'écart spectral varie. C'est un fait sur l'instrument, indépendant de
Kochen–Specker, et il mérite son propre examen — il n'est pas traité ici.

### Exercice 3 — poser la question rendue impossible

C'est l'exercice ouvert, et le seul dont la réponse n'est pas connue.

L'annexe conclut que le zoo manque d'une structure de co-mesurabilité **partielle**.
Propose-en une, puis teste-la.

Un candidat sérieux doit répondre par écrit à : *pourquoi ces deux proxys ne peuvent-ils
pas être calculés sur la même trajectoire ?* Une réponse acceptable invoque une
incompatibilité de **prétraitement** — par exemple deux discrétisations, deux fenêtrages ou
deux normalisations qu'aucun pipeline ne peut appliquer simultanément à la même série.

Attention au piège que cette annexe vient de documenter : si ta structure produit un
recouvrement unique, le garde rendra `degenerate_single_cover` et la question sera de
nouveau non posable. Il faut un recouvrement **partiel** en cycle.

In [8]:
#nullable enable
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Exercice 3 — proposer une structure de co-mesurabilite partielle JUSTIFIEE.
//
// TODO etudiant : proposer une structure de co-mesurabilite partielle JUSTIFIEE.
//
// Indice 1 : la justification est la partie difficile, et elle est textuelle.
//            Ecris-la en commentaire ci-dessous AVANT de coder quoi que ce soit ;
//            un decoupage arbitraire des proxys en contextes ne demontrerait rien
//            (ce serait exactement l'analogie decorative que l'annexe refuse).
// Indice 2 : la structure viable est un cycle -- A={p,q}, B={q,r}, C={r,p} --
//            avec p, q, r deux a deux co-mesurables mais jamais les trois ensemble.
// Indice 3 : verifie d'abord Csp.Rapport(modele).NRecouvrementsDistincts > 1
//            AVANT de regarder le verdict. Si le recouvrement est unique,
//            la question n'est pas posee et le verdict ne veut rien dire.
//
// Ma justification (a rediger) :
//   proxys retenus  : ...
//   incompatibilite : ...
//   pourquoi elle n'est pas un artefact de mise en oeuvre : ...

List<Contexte>? structure = null;

// Etape 1 : construire le modele a partir de la structure proposee
// var modele = ModeleEmpirique.FromContexts(structure!, new[] { 0, 1 });

// Etape 2 : verifier le recouvrement AVANT le verdict
// var rapport = Csp.Rapport(modele);
// rapport.NRecouvrementsDistincts > 1

// Etape 3 : rendre le verdict et l'interpreter avec les trois crans
// var resultat = Csp.Verdict(modele);

Console.WriteLine("Exercice 3 a completer");

// Attestation de parite Python/CSharp : les valeurs attendues ci-dessous sont
// les valeurs de reference du carnet Python, comparees a la precision F4
// affichee par le jumeau. Elles ne dependent pas de la solution de l'exercice.
var referencePython = new Dictionary<string, string[]>(StringComparer.Ordinal)
{
    ["coarse_graining_k3"] = new[]
    {
        "seed= 0  gap=0.0193  sens_mean=1.3333  sens_max=2.0000",
        "seed= 1  gap=0.0195  sens_mean=1.3333  sens_max=2.0000",
        "seed= 7  gap=0.0135  sens_mean=1.3333  sens_max=2.0000",
        "seed=42  gap=0.0230  sens_mean=1.3333  sens_max=2.0000",
        "seed=99  gap=0.0238  sens_mean=1.3333  sens_max=2.0000"
    },
    ["coarse_graining_k4"] = new[]
    {
        "seed= 0  gap=0.0134  sens_mean=1.5000  sens_max=2.0000",
        "seed= 1  gap=0.0133  sens_mean=1.5000  sens_max=2.0000",
        "seed= 7  gap=0.0115  sens_mean=1.5000  sens_max=2.0000",
        "seed=42  gap=0.0169  sens_mean=1.5000  sens_max=2.0000",
        "seed=99  gap=0.0172  sens_mean=1.5000  sens_max=2.0000"
    },
    ["coarse_graining_k6"] = new[]
    {
        "seed= 0  gap=0.0085  sens_mean=1.6667  sens_max=2.0000",
        "seed= 1  gap=0.0098  sens_mean=1.6667  sens_max=2.0000",
        "seed= 7  gap=0.0067  sens_mean=1.6667  sens_max=2.0000",
        "seed=42  gap=0.0110  sens_mean=1.6667  sens_max=2.0000",
        "seed=99  gap=0.0121  sens_mean=1.6667  sens_max=2.0000"
    }
};

int nbValeursComparees = 0;
bool parite = true;
foreach (var kv in referencePython)
{
    var runs = signatures[kv.Key];
    for (int i = 0; i < runs.Count; i++)
    {
        var r = runs[i];
        string ligneCSharp =
            "seed=" + SEEDS[i].ToString(CultureInfo.InvariantCulture).PadLeft(2)
            + "  gap=" + r["spectral_gap"].ToString("F4", CultureInfo.InvariantCulture)
            + "  sens_mean=" + r["sensitivity_mean"].ToString("F4", CultureInfo.InvariantCulture)
            + "  sens_max=" + r["sensitivity_max"].ToString("F4", CultureInfo.InvariantCulture);
        string ligneReference = kv.Value[i];
        bool egal = ligneCSharp == ligneReference;
        parite &= egal;
        nbValeursComparees++;
        Verifs.Vrai("valeur Python/CSharp — " + kv.Key + " / graine "
            + SEEDS[i].ToString(CultureInfo.InvariantCulture), egal);
    }
}

Verifs.Entier("nombre de valeurs Python/CSharp comparees", nbValeursComparees, 15);
Verifs.Vrai("attestation de parite Python/CSharp sur toutes les signatures", parite);
Verifs.Vrai("seuils Python/CSharp — medianes coincidentes",
    seuils["spectral_gap"].ToString("F6", CultureInfo.InvariantCulture) == "0.013437"
    && seuils["sensitivity_mean"].ToString("F6", CultureInfo.InvariantCulture) == "1.500000"
    && seuils["sensitivity_max"].ToString("F6", CultureInfo.InvariantCulture) == "2.000000");
Console.WriteLine("PARITE Python/CSharp — CONFORME : "
    + nbValeursComparees.ToString(CultureInfo.InvariantCulture)
    + " signatures comparees a la precision F4, seuils F6 et verdict final");

Exercice 3 a completer


CONFORME — valeur Python/CSharp — coarse_graining_k3 / graine 0


CONFORME — valeur Python/CSharp — coarse_graining_k3 / graine 1


CONFORME — valeur Python/CSharp — coarse_graining_k3 / graine 7


CONFORME — valeur Python/CSharp — coarse_graining_k3 / graine 42


CONFORME — valeur Python/CSharp — coarse_graining_k3 / graine 99


... (10 outputs left out)

CONFORME — nombre de valeurs Python/CSharp comparees : calcule = 15 | attendu = 15


CONFORME — attestation de parite Python/CSharp sur toutes les signatures


CONFORME — seuils Python/CSharp — medianes coincidentes


PARITE Python/CSharp — CONFORME : 15 signatures comparees a la precision F4, seuils F6 et verdict final


## 5. Où l'analogie meurt — quatre points de rupture

Un transfert non instruit *est* l'analogie décorative. Ces quatre points sont les endroits
où le cadre quantique et le zoo de proxys cessent de se correspondre, et ils doivent
accompagner tout usage de cette machinerie.

1. **Origine des contraintes.** Chez Kochen–Specker, les contraintes sont **algébriques**
   (relations d'orthogonalité entre projecteurs) : elles sont exactes. Ici, elles sont
   **mesurées** — un support est un ensemble de tuples *observés*. Un `UNSAT` peut donc
   être un artefact d'échantillonnage plutôt qu'une obstruction.

2. **Identité des observables.** En quantique, « le même observable dans deux contextes »
   est littéralement *le même opérateur*. Ici, un proxy calculé sous deux grains de
   description n'est pas garanti être la même grandeur. C'est **l'hypothèse la plus forte**
   du transfert, et elle n'est pas démontrée.

3. **Pas de théorème de dimension.** Kochen–Specker exige $\dim \geq 3$ et des
   configurations de rayons précises ; rien de tel ne contraint le zoo. Conséquence
   symétrique et souvent oubliée : l'absence d'obstruction ici n'a **aucune portée** sur
   le théorème de Kochen–Specker, ni réciproquement.

4. **Discrétisation.** Les seuils qui transforment des valeurs continues en issues
   discrètes sont un **choix**. Un verdict doit être rapporté avec sa sensibilité à ce
   choix, sans quoi il mesure le seuil autant que le substrat.

## Vérifiez votre compréhension

1. La section 1 valide le détecteur sur un cas prouvé à la main avant de le croire — pourquoi ce contrôle positif est-il un prérequis au braquage sur le zoo ?
2. La section 3 énumère trois manières d'être vide — pourquoi un `UNSAT` du zoo ne peut-il pas être lu comme une obstruction ?
3. La section 5 désigne l'identité des observables comme l'hypothèse la plus forte du transfert — pourquoi n'est-elle pas démontrée ?

## À retenir

*La contribution de cette annexe n'est pas un verdict de contextualité — il n'y en a pas.
C'est une méthode :*

- une question conceptuelle importée peut devenir un objet calculable, et c'est ce qui la
  rend contestable plutôt que décorative ;
- un détecteur se **valide sur un cas prouvé à la main** avant d'être braqué sur des
  données ;
- et surtout : **un verdict ne vaut que si sa négation était atteignable.** Les trois
  dégénérescences ne sont pas un raffinement cosmétique — ce sont elles qui ont transformé
  un faux positif séduisant en un résultat négatif utilisable.

> **Note de grade C** (discipline #8182) : la lecture de la contextualité comme
> « obstruction au recollement » circule dans la littérature adjacente (*boundary problem*).
> Elle est ici un **hook documentaire**, jamais créditée comme résultat.

## Pour aller plus loin

Issue #7290 (registre de cette annexe) · module `ict/proxy_contextuality.py`
et sa suite de tests · `docs/ict/synthese-invariants-dissociations-obstructions.md`
(section dédiée) · [ICT-15d](ICT-15-IntegratedComplexity/ICT-15d-CechObstruction-Python.ipynb) pour la leçon d'origine sur les verdicts vides.

- **[ICT-0 — Cadrage de la série](ICT-0-Framing.md)** : le principe directeur de la série ICT.

## Attestation de parité Python/CSharp

La cellule de code immédiatement précédente est l'**attestation de comparaison** du
jumeau : elle confronte les 15 signatures produites en C# aux valeurs de référence du
carnet Python, à la précision `F4`, puis vérifie les trois médianes à la précision `F6`
et le verdict final. Sa sortie doit se terminer par :

`PARITE Python/CSharp — CONFORME`

Cette attestation est distincte de la provenance `Carnet issu de l'EPIC #26534` :
elle
compare les instances et les valeurs effectivement produites par les deux jumeaux. Les
contrôles `CONFORME` des cellules précédentes restent des contrôles locaux ; ce sont les
contrôles d'attestation de cette section qui établissent la parité finale.

> Carnet issu de l'EPIC #26534.